# Plates detection

## Libraries & Dataset

In [1]:
!pip install -qU roboflow ultralytics wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 67.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.8 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("RoboFlow")
wandb_api_key = user_secrets.get_secret("WandB_SafeSpace")

wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmed-hossam (ahmed-hossam-suez-canal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(13)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-13 in yolo26:: 100%|██████████| 203744/203744 [00:22<00:00, 9156.98it/s] 


In [7]:
# ── W&B: Initialize run with full hyperparameter config ──────────────
EPOCHS = 12
IMGSZ  = 640
BATCH  = 16
MODEL  = "yolo26l.pt"
PROJECT = "License_Plate_OCR"
RUN_NAME = "v5_plate_detection"

run = wandb.init(
    project=PROJECT,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL,
        "pretrained":   True,
        # Training
        "epochs":          EPOCHS,
        "imgsz":           IMGSZ,
        "batch":           BATCH,
        "fraction":        0.20,     # Only use 20% of the data
        "optimizer":       "auto",
        "lr0":             0.01,
        "lrf":             0.01,
        "momentum":        0.937,
        "weight_decay":    0.0005,
        "warmup_epochs":   3.0,
        "cos_lr":          True,
        "patience":        10,
        "box":             9.0,
        # Augmentation — stripped to avoid double-augmenting Roboflow's offline transforms
        "fliplr":          0.0,    # Roboflow already flipped
        "hsv_h":           0.0,    # Roboflow already shifted hue
        "hsv_s":           0.0,    # Roboflow already shifted saturation
        "hsv_v":           0.0,    # Roboflow already shifted brightness/exposure
        "erasing":         0.0,    # Roboflow already applied cutout
        "mosaic":          1.0,    # ✅ Keep — not in Roboflow pipeline
        "translate":       0.1,    # ✅ Keep — adds useful position variance
        "scale":           0.1,    # ✅ Keep — different from Roboflow's fixed crop
        # Dataset
        "dataset":         "license-plate-recognition-rxg4e",
        "dataset_version": 13,
        "dataset_link":    "https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e/dataset/13",
        "num_classes":     1,
}
)
print(f"W&B run started: {run.url}")


W&B run started: https://wandb.ai/ahmed-hossam-suez-canal-university/License_Plate_OCR/runs/j39hqbch


## Modeling

In [8]:
from ultralytics import YOLO

cfg = wandb.config  # use values logged to W&B

model = YOLO(cfg.model)

results = model.train(
    data='/kaggle/working/License-Plate-Recognition-13/data.yaml',
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    fraction=cfg.fraction,
    cos_lr=cfg.cos_lr,
    patience=cfg.patience,
    box=cfg.box,
    erasing=cfg.erasing,
    fliplr=cfg.fliplr,
    hsv_h=cfg.hsv_h,
    hsv_s=cfg.hsv_s,
    hsv_v=cfg.hsv_v,
    scale=cfg.scale,
    project=PROJECT,
    name=RUN_NAME,
    plots=True,
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=9, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/License-Plate-Recognition-13/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=12, erasing=0, exist_ok=False, fliplr=0, flipud=0.0, format=torchscript, fraction=0.2, freeze=None, half=False, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v5_plate_detection2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

In [9]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/box_loss":     metrics_dict.get("val/box_loss",         0),
    "final/cls_loss":     metrics_dict.get("val/cls_loss",         0),
    "final/dfl_loss":     metrics_dict.get("val/dfl_loss",         0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


Logged metrics:
  final/precision: 0.8977
  final/recall: 0.8436
  final/mAP50: 0.8962
  final/mAP50-95: 0.5358
  final/box_loss: 0.0000
  final/cls_loss: 0.0000
  final/dfl_loss: 0.0000
  final/fitness: 0.5358


In [10]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


  ✓ confusion_matrix
  ✓ confusion_matrix_normalized
  ✓ BoxPR_curve
  ✓ BoxF1_curve
  ✓ BoxP_curve
  ✓ BoxR_curve
  ✓ results
  ✓ labels
  ✓ val_batch0_labels
  ✓ val_batch0_pred
  ✓ val_batch1_labels
  ✓ val_batch1_pred
  ✓ val_batch2_labels
  ✓ val_batch2_pred

Logged 14 images/plots to W&B.


In [13]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name="license_plate_detector",
    type="model",
    description="YOLOv26l fine-tuned for license plate detection",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
        "dataset_link": cfg.dataset_link,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


Model artifact logged: license_plate_detector:v1:v1


In [14]:
# Finish the WandB run
wandb.finish()

final/box_loss,▁
final/cls_loss,▁
final/dfl_loss,▁
final/fitness,▁
final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/box_loss,0
final/cls_loss,0
final/dfl_loss,0
